In [1]:
print("Here begins the Bayesian networks notebook.")

Here begins the Bayesian networks notebook.


In [3]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
df_bn = pd.read_csv("../../data/processed/student_depression_bn_clean.csv")

df_bn.head()

,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,male,33.0,visakhapatnam,student,5.0,0.0,8.97,2.0,0.0,5-6 hours,Healthy,b.pharm,yes,3.0,1.0,no,yes
1,female,24.0,bangalore,student,2.0,0.0,5.90,5.0,0.0,5-6 hours,Moderate,bsc,no,3.0,2.0,yes,no
2,male,31.0,srinagar,student,3.0,0.0,7.03,5.0,0.0,Less than 5 hours,Healthy,ba,no,9.0,1.0,yes,no
3,female,28.0,varanasi,student,3.0,0.0,5.59,2.0,0.0,7-8 hours,Moderate,bca,yes,4.0,5.0,yes,yes
4,female,25.0,jaipur,student,4.0,0.0,8.13,3.0,0.0,5-6 hours,Moderate,m.tech,yes,1.0,1.0,no,no


In [4]:
# Inspect dataset shape, column types, and missing values
print(df_bn.shape)
print(df_bn.dtypes)

print("\nMissing values per column:")
print(df_bn.isna().sum().sort_values(ascending=False))

(27901, 17)
Gender                                       str
Age                                      float64
City                                         str
Profession                                   str
Academic Pressure                        float64
Work Pressure                            float64
CGPA                                     float64
Study Satisfaction                       float64
Job Satisfaction                         float64
Sleep Duration                               str
Dietary Habits                               str
Degree                                       str
Have you ever had suicidal thoughts ?        str
Work/Study Hours                         float64
Financial Stress                         float64
Family History of Mental Illness             str
Depression                                   str
dtype: object

Missing values per column:
Financial Stress                         3
Gender                                   0
Age                        

In [5]:
# Convert all columns to string
# Missing values are preserved as NaN
for col in df_bn.columns:
    df_bn[col] = df_bn[col].astype("string")

print(df_bn.dtypes)

Gender                                   string
Age                                      string
City                                     string
Profession                               string
Academic Pressure                        string
Work Pressure                            string
CGPA                                     string
Study Satisfaction                       string
Job Satisfaction                         string
Sleep Duration                           string
Dietary Habits                           string
Degree                                   string
Have you ever had suicidal thoughts ?    string
Work/Study Hours                         string
Financial Stress                         string
Family History of Mental Illness         string
Depression                               string
dtype: object


In [6]:
# Check the number of unique states per variable
cardinality = pd.DataFrame({
    "Variable": df_bn.columns,
    "Num_States": [df_bn[col].nunique(dropna=True) for col in df_bn.columns]
}).sort_values("Num_States", ascending=False)

cardinality

,Variable,Num_States
6,CGPA,332
2,City,52
1,Age,34
11,Degree,28
3,Profession,14
13,Work/Study Hours,13
4,Academic Pressure,6
7,Study Satisfaction,6
14,Financial Stress,5
9,Sleep Duration,5


In [7]:
# Discretize CGPA into 4–5 bins
df_bn["CGPA_binned"] = pd.qcut(
    pd.to_numeric(df_bn["CGPA"], errors="coerce"),
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop"
).astype("string")

# Drop original CGPA and City
df_bn = df_bn.drop(columns=["CGPA", "City"])
df_bn_model = df_bn.copy()

print(df_bn_model.shape)
print(df_bn_model.columns.tolist())

(27901, 16)
['Gender', 'Age', 'Profession', 'Academic Pressure', 'Work Pressure', 'Study Satisfaction', 'Job Satisfaction', 'Sleep Duration', 'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?', 'Work/Study Hours', 'Financial Stress', 'Family History of Mental Illness', 'Depression', 'CGPA_binned']


In [8]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pgmpy.estimators import HillClimbSearch, ExpertKnowledge, BayesianEstimator
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

In [9]:
# Unconstrained Bayesian Network structure using Hill-Climb Search
# - bic-d - used for categorical data
# - max_indegree - the number of parents per node is limited to avoid dense graphs
hc = HillClimbSearch(df_bn_model)

learned_dag = hc.estimate(
    scoring_method="bic-d",
    max_indegree=3,
    show_progress=False
)

print("Number of nodes:", len(learned_dag.nodes()))
print("Number of edges:", len(learned_dag.edges()))
print("\nLearned edges:")
print(sorted(learned_dag.edges()))

Number of nodes: 16
Number of edges: 13

Learned edges:
[('Academic Pressure', 'Study Satisfaction'), ('Age', 'Degree'), ('Depression', 'Academic Pressure'), ('Depression', 'Age'), ('Depression', 'Dietary Habits'), ('Depression', 'Family History of Mental Illness'), ('Depression', 'Financial Stress'), ('Depression', 'Sleep Duration'), ('Depression', 'Study Satisfaction'), ('Depression', 'Work/Study Hours'), ('Dietary Habits', 'Gender'), ('Gender', 'CGPA_binned'), ('Have you ever had suicidal thoughts ?', 'Depression')]


In [10]:
# Convert the learned graph into a discrete Bayesian Network
bn_unconstrained = DiscreteBayesianNetwork(learned_dag.edges())

# Fit CPDs through Bayesian parameter estimation
# BDeu prior avoids zero probabilities
bn_unconstrained.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Model check:", bn_unconstrained.check_model())

Model check: True


In [16]:
# conditional probability distribution of the target variable
# how Depression depends on learned parent variables

cpd_dep = bn_unconstrained.get_cpds("Depression")

# Extract state names
dep_states = cpd_dep.state_names["Depression"]
parent = list(cpd_dep.state_names.keys())[1]
parent_states = cpd_dep.state_names[parent]

# Values
values = cpd_dep.values

# DataFrame
cpd_df = pd.DataFrame(
    values,
    index=[f"Depression = {s}" for s in dep_states],
    columns=[f"{parent} = {s}" for s in parent_states]
)

cpd_df

,Have you ever had suicidal thoughts ? = no,Have you ever had suicidal thoughts ? = yes
Depression = no,0.767659,0.209586
Depression = yes,0.232341,0.790414


In [17]:
# Create inference engine for the unconstrained BN
infer_unconstrained = VariableElimination(bn_unconstrained)  # type: ignore

In [21]:
# Query 1:
# Probability of Depression given high academic pressure and suicidal thoughts
q1 = infer_unconstrained.query(
    variables=["Depression"],
    evidence={  # type: ignore
        "Academic Pressure": "5.0",
        "Have you ever had suicidal thoughts ?": "yes"
    },
    show_progress=False
)

print(q1)

+-----------------+-------------------+
| Depression      |   phi(Depression) |
+=================+===================+
| Depression(no)  |            0.0571 |
+-----------------+-------------------+
| Depression(yes) |            0.9429 |
+-----------------+-------------------+


In [22]:
# Query 2:
# Probability of Depression under poor sleep, unhealthy diet, and financial stress
q2 = infer_unconstrained.query(
    variables=["Depression"],
    evidence={  # type: ignore
        "Sleep Duration": "Less than 5 hours",
        "Dietary Habits": "Unhealthy",
        "Financial Stress": "5.0"
    },
    show_progress=False
)

print(q2)

+-----------------+-------------------+
| Depression      |   phi(Depression) |
+=================+===================+
| Depression(no)  |            0.0948 |
+-----------------+-------------------+
| Depression(yes) |            0.9052 |
+-----------------+-------------------+


In [25]:
# Query 3:
# Depression probabilities under two contrasting scenarios

q3_supportive = infer_unconstrained.query(
    variables=["Depression"],
    evidence={  # type: ignore
        "Study Satisfaction": "5.0",
        "Sleep Duration": "7-8 hours",
        "Financial Stress": "1.0"
    },
    show_progress=False
)

q3_risky = infer_unconstrained.query(
    variables=["Depression"],
    evidence={  # type: ignore
        "Study Satisfaction": "1.0",
        "Sleep Duration": "Less than 5 hours",
        "Financial Stress": "5.0"
    },
    show_progress=False
)

print("Depression distribution under supportive conditions:")
print(q3_supportive)

print("\nDepression distribution under unsupportive conditions:")
print(q3_risky)

Depression distribution under supportive conditions:
+-----------------+-------------------+
| Depression      |   phi(Depression) |
+=================+===================+
| Depression(no)  |            0.7644 |
+-----------------+-------------------+
| Depression(yes) |            0.2356 |
+-----------------+-------------------+

Depression distribution under risky conditions:
+-----------------+-------------------+
| Depression      |   phi(Depression) |
+=================+===================+
| Depression(no)  |            0.0947 |
+-----------------+-------------------+
| Depression(yes) |            0.9053 |
+-----------------+-------------------+


In [26]:
# Build a restricted BN with simple expert constraints- prevent obviously implausible directions

forbidden_edges = [
    # Depression should not cause background variables
    ("Depression", "Gender"),
    ("Depression", "Age"),
    ("Depression", "City"),
    ("Depression", "Profession"),
    ("Depression", "Degree"),
    ("Depression", "Family History of Mental Illness"),

    # Depression should not cause broad structural or background study/work context
    ("Depression", "Academic Pressure"),
    ("Depression", "Work Pressure"),
    ("Depression", "CGPA"),
    ("Depression", "Study Satisfaction"),
    ("Depression", "Job Satisfaction"),
    ("Depression", "Sleep Duration"),
    ("Depression", "Dietary Habits"),
    ("Depression", "Work/Study Hours"),
    ("Depression", "Financial Stress"),

    # Suicidal thoughts should not cause background variables
    ("Have you ever had suicidal thoughts?", "Gender"),
    ("Have you ever had suicidal thoughts?", "Age"),
    ("Have you ever had suicidal thoughts?", "Family History of Mental Illness"),
]

expert_knowledge = ExpertKnowledge(
    forbidden_edges=forbidden_edges
)

hc_restricted = HillClimbSearch(df_bn_model)

learned_dag_restricted = hc_restricted.estimate(
    scoring_method="bic-d",
    expert_knowledge=expert_knowledge,
    max_indegree=3,
    show_progress=False
)

print("Restricted BN edges:")
print(sorted(learned_dag_restricted.edges()))

Restricted BN edges:
[('Academic Pressure', 'Depression'), ('Academic Pressure', 'Financial Stress'), ('Age', 'Degree'), ('Depression', 'Have you ever had suicidal thoughts ?'), ('Dietary Habits', 'Gender'), ('Financial Stress', 'Depression'), ('Gender', 'CGPA_binned'), ('Have you ever had suicidal thoughts ?', 'Age'), ('Have you ever had suicidal thoughts ?', 'Dietary Habits'), ('Have you ever had suicidal thoughts ?', 'Family History of Mental Illness'), ('Have you ever had suicidal thoughts ?', 'Sleep Duration'), ('Have you ever had suicidal thoughts ?', 'Work/Study Hours'), ('Study Satisfaction', 'Academic Pressure')]


In [27]:
# Fit the restricted BN
bn_restricted = DiscreteBayesianNetwork(learned_dag_restricted.edges())

bn_restricted.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Restricted model check:", bn_restricted.check_model())

Restricted model check: True


In [28]:
# Compare the direct parent variables of Depression in the restricted BN
if "Depression" in bn_restricted.nodes():
    print("Parents of Depression in restricted BN:")
    print(list(bn_restricted.predecessors("Depression")))

Parents of Depression in restricted BN:
['Academic Pressure', 'Financial Stress']


In [29]:
# Create inference engine for the restricted BN
infer_restricted = VariableElimination(bn_restricted)  # type: ignore

In [31]:
# Query 4:
# Probability of Depression given suicidal thoughts and family history
q4 = infer_restricted.query(
    variables=["Depression"],
    evidence={  # type: ignore
        "Have you ever had suicidal thoughts ?": "yes",
        "Family History of Mental Illness": "yes"
    },
    show_progress=False
)

print(q4)

+-----------------+-------------------+
| Depression      |   phi(Depression) |
+=================+===================+
| Depression(no)  |            0.2096 |
+-----------------+-------------------+
| Depression(yes) |            0.7904 |
+-----------------+-------------------+


In [32]:
# Query 5:
# Probability of Depression under strong academic and financial pressure
q5 = infer_restricted.query(
    variables=["Depression"],
    evidence={  # type: ignore
        "Academic Pressure": "5.0",
        "Financial Stress": "5.0",
        "Study Satisfaction": "1.0"
    },
    show_progress=False
)

print(q5)

+-----------------+-------------------+
| Depression      |   phi(Depression) |
+=================+===================+
| Depression(no)  |            0.0459 |
+-----------------+-------------------+
| Depression(yes) |            0.9541 |
+-----------------+-------------------+


In [33]:
# Summarize the local neighborhood of the target variable
summary_rows = []

if "Depression" in bn_restricted.nodes():
    summary_rows.append({
        "Target": "Depression",
        "Parents": list(bn_restricted.predecessors("Depression")),
        "Children": list(bn_restricted.successors("Depression"))
    })

pd.DataFrame(summary_rows)

,Target,Parents,Children
0,Depression,"[Academic Pressure, Financial Stress]",[Have you ever had suicidal thoughts ?]


In [34]:
# Inspect the CPD of Depression in the restricted model
if "Depression" in bn_restricted.nodes():
    print("Restricted CPD for Depression:")
    print(bn_restricted.get_cpds("Depression"))

Restricted CPD for Depression:
+-------------------+-----+------------------------+
| Academic Pressure | ... | Academic Pressure(5.0) |
+-------------------+-----+------------------------+
| Financial Stress  | ... | Financial Stress(5.0)  |
+-------------------+-----+------------------------+
| Depression(no)    | ... | 0.04591216802638431    |
+-------------------+-----+------------------------+
| Depression(yes)   | ... | 0.9540878319736157     |
+-------------------+-----+------------------------+
